# FileSessionManager — Local Persistence

The `FileSessionManager` stores conversation state as JSON files on the local filesystem.
Each session gets its own directory with agent metadata and individual message files.

## Filesystem Structure

```
/<storage_dir>/
└── session_<session_id>/
    ├── session.json
    └── agents/
        └── agent_<agent_id>/
            ├── agent.json
            └── messages/
                ├── message_0.json
                └── message_1.json
```

In [ ]:
%pip install -q --upgrade strands-agents

## Create an agent with FileSessionManager

In [ ]:
from strands import Agent
from strands.session.file_session_manager import FileSessionManager

SESSION_ID = "my-first-session"
STORAGE_DIR = "./sessions"

session_manager = FileSessionManager(
    session_id=SESSION_ID,
    storage_dir=STORAGE_DIR,
)

agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=session_manager,
)

# Share some information
response = agent("My name is Alex and I'm building a chatbot for my startup.")
print(response)

In [ ]:
# Continue the conversation
response = agent("The startup is called BrightBot and we focus on customer support automation.")
print(response)

## Inspect the persisted data

Let's look at what FileSessionManager wrote to disk.

In [ ]:
import os
import json

# Walk the session directory
for root, dirs, files in os.walk(STORAGE_DIR):
    level = root.replace(STORAGE_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

In [ ]:
# Read the session metadata
session_file = os.path.join(STORAGE_DIR, f"session_{SESSION_ID}", "session.json")
with open(session_file) as f:
    session_data = json.load(f)
print(json.dumps(session_data, indent=2))

## Restore the session after a "restart"

Create a new agent with the same session ID. The FileSessionManager will
load the persisted conversation automatically.

In [ ]:
# Simulate restart — new agent, same session ID
restored_session_manager = FileSessionManager(
    session_id=SESSION_ID,
    storage_dir=STORAGE_DIR,
)

restored_agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=restored_session_manager,
)

# The agent remembers the previous conversation
response = restored_agent("What's my name and what company do I work for?")
print(response)

The agent recalls Alex and BrightBot from the persisted session.

## Cleanup

In [ ]:
import shutil

# Remove the local session files
shutil.rmtree(STORAGE_DIR, ignore_errors=True)
print("Session files cleaned up.")